# Driver and Safety Analysis

## Goal

1. Who are the drivers, and how long have they been with the company?
2. Does on-time rate or fuel efficiency differ by driver tenure/experience?
3. Do individual drivers stand out, or is the spread consistent with chance?
4. What do safety incident rates look like, and do they relate to driver experience?

## Data Basis

`drivers`, `driver_monthly_metrics`, `trips`, `fuel_purchases` (only for the `driver_id` NULL check),
`safety_incidents`, plus delivery metrics
from the pipeline. Note: `safety_incidents` has only 170 records - treat rates and
comparisons with appropriate caution given the small sample.

## 1. Load Data (via Pipeline)

In [ ]:
import sys

sys.path.append("../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.getLogger("matplotlib.category").setLevel(logging.WARNING)

sns.set_theme(style="darkgrid")

from delivery_pipeline import PROCESSED_DATA_PATH_DRIVER, RAW_DATA_DIR, run_pipeline

clean_df = run_pipeline(output_path=PROCESSED_DATA_PATH_DRIVER, include_drivers=True)
drivers_df = pd.read_csv(RAW_DATA_DIR / "drivers.csv")
driver_monthly_metrics_df = pd.read_csv(RAW_DATA_DIR / "driver_monthly_metrics.csv")
safety_incidents_df = pd.read_csv(RAW_DATA_DIR / "safety_incidents.csv")

print("clean_df:", clean_df.shape)
print("drivers_df:", drivers_df.shape)
print("driver_monthly_metrics_df:", driver_monthly_metrics_df.shape)
print("safety_incidents_df:", safety_incidents_df.shape)

## 2. Data Quality Checks

In [ ]:
# NULL share of driver_id in the load table (expected ~2%, intentional per dataset description)
print(f"Loads without driver_id: {clean_df['driver_id'].isna().mean():.4%}")

# fuel_purchases is in the Data Basis only for this check
fuel = pd.read_csv(RAW_DATA_DIR / "fuel_purchases.csv")
print(f"Fuel purchases without driver_id: {fuel['driver_id'].isna().mean():.4%}")

# Orphans and duplicates
known = drivers_df["driver_id"]
print(
    "Loads with unknown driver_id:",
    (clean_df["driver_id"].notna() & ~clean_df["driver_id"].isin(known)).sum(),
)
print("Duplicate driver_ids in drivers:", known.duplicated().sum())
print(
    "Incidents with unknown driver_id:",
    (
        safety_incidents_df["driver_id"].notna()
        & ~safety_incidents_df["driver_id"].isin(known)
    ).sum(),
)
print(
    "Incidents without driver_id / truck_id:",
    safety_incidents_df["driver_id"].isna().sum(),
    safety_incidents_df["truck_id"].isna().sum(),
)

In [ ]:
# trip_id can be NaN for loads without a trip; drop those so that the right side is unique
trip_info = clean_df.dropna(subset=["trip_id"])[["trip_id", "driver_id", "truck_id"]]
linked = safety_incidents_df.merge(
    trip_info,
    on="trip_id",
    how="left",
    suffixes=("", "_trip"),
    validate="many_to_one",
    indicator=True,
)
print(
    linked["_merge"].value_counts()
)  # how many incidents link to a trip in the cleaned data?

both = (
    linked["_merge"].eq("both")
    & linked["driver_id"].notna()
    & linked["driver_id_trip"].notna()
)
print(
    "driver_id matches the trip:",
    (linked.loc[both, "driver_id"] == linked.loc[both, "driver_id_trip"]).mean(),
)
both_t = (
    linked["_merge"].eq("both")
    & linked["truck_id"].notna()
    & linked["truck_id_trip"].notna()
)
print(
    "truck_id matches the trip:",
    (linked.loc[both_t, "truck_id"] == linked.loc[both_t, "truck_id_trip"]).mean(),
)

In [ ]:
no_driver = safety_incidents_df[safety_incidents_df["driver_id"].isna()]
print(no_driver[["incident_id", "trip_id", "truck_id", "driver_id"]])

# does the linked trip have a driver_id? (trip_info comes from the cell above)
print(no_driver["trip_id"].map(trip_info.set_index("trip_id")["driver_id"]))

### Data Quality Note

Checked: share of loads without `driver_id`, orphan and duplicate keys, and whether the incidents link to trips of the cleaned data (84,924 loads). 

Result: 2.01 % of the cleaned loads have no `driver_id` (the dataset description states about 2 % as intentional); there are no unknown `driver_id`s in the loads or the incidents and no duplicates in `drivers`. All 170 incidents link to a trip of the cleaned data. One incident has neither `driver_id` nor `truck_id`; for the other 169 incidents, `driver_id` and `truck_id` match the linked trip in 100 % of cases. Consequence: driver analyses exclude the loads without `driver_id`; incident rates use all 170 incidents and the miles of the same trips, and the tenure of an incident is taken from its trip. The incident without `driver_id` is assigned via its trip.

## 3. Driver Overview

### 3.1 Tenure at trip date

*Note: tenure is measured at the time of each trip (as `pickup_actual_datetime` - `hire_date`), not as of today, because the data covers 2022-2024.*

In [ ]:
# 3.1 Tenure in years at the time of the trip (no rounding)
seconds_per_year = 365.25 * 24 * 3600
clean_df["tenure_years"] = (
    clean_df["pickup_actual_datetime"] - clean_df["hire_date"]
).dt.total_seconds() / seconds_per_year

print("Trips with negative tenure:", (clean_df["tenure_years"] < 0).sum())
print(
    "Trips after the termination date:",
    (clean_df["pickup_actual_datetime"] > clean_df["termination_date"]).sum(),
)
print(clean_df["tenure_years"].describe())

### 3.2 Driver distribution


In [ ]:
# 3.2 Driver distribution
hire = pd.to_datetime(drivers_df["hire_date"])
term = pd.to_datetime(drivers_df["termination_date"])
print(hire.dt.year.value_counts().sort_index())  # hires per year
print("Still employed:", term.isna().sum(), "| terminated:", term.notna().sum())
print("Drivers in clean_df: ", clean_df["driver_id"].nunique())
print(
    "Trips per driver: \n", clean_df.groupby("driver_id").size().describe()
)  # trips per driver
drivers_with_trips = clean_df.drop_duplicates("driver_id").dropna(subset=["driver_id"])
print(drivers_with_trips["cdl_class"].value_counts())
print(drivers_with_trips["years_experience"].describe())

In [ ]:
years = range(hire.dt.year.min(), hire.dt.year.max() + 1)
hires_per_year = hire.dt.year.value_counts().reindex(years, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(x=hires_per_year.index, y=hires_per_year.values, label="Hires", ax=ax)
ax.bar_label(ax.containers[0], fmt="%.0f", padding=3)
ax.axhline(0, color="black", linewidth=0.8)
ax.margins(y=0.15)
ax.set_xlabel("Hire Year")
ax.set_ylabel("Hire Count")
ax.set_title("Hires per Year")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
trips_per_driver = clean_df["driver_id"].value_counts()  # one value per driver

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(trips_per_driver, bins="auto", kde=True, ax=ax)
ax.axvline(
    trips_per_driver.mean(),
    color="red",
    linestyle="--",
    label=f"Avg. trips per driver: {trips_per_driver.mean():.2f}",
)
ax.set_xlabel("Trips per Driver")
ax.set_ylabel("Number of Drivers")
ax.set_title("Distribution of Trips per Driver")
ax.legend()
plt.show()

#### Interpretation
Checked: who the drivers are and how much tenure the trips cover. 
Result: the table holds 150 drivers, but only the 124 drivers still employed appear in the trips; the 26 terminated drivers have no trips. All drivers were hired between 2012 and 2021 (8–21 per year), none during the data window, so tenure at the trip date ranges from 0.08 to 12.9 years (median 5.95). No trip has a negative tenure or lies after a termination date. Trips are spread evenly across drivers (601–745 each, mean 671, standard deviation 29, about 4 %), and all drivers hold CDL class A.

Classification: the tenure range is wide enough to compare groups in section 4, and the even workload means that no single driver dominates a group. Because all drivers hold the same CDL class, it cannot explain differences and is not used as a variable.

Consequence: section 4 compares drivers by tenure at the trip date. Since drivers who left have no trips, the results only describe drivers who stayed at the company and say nothing about whether newer drivers leave earlier.

Idea for later (not a finding): years_experience (range 2–25 years) is a second measure besides tenure. If it turns out to be independent of tenure, it could be examined the same way as tenure, for example as an optional section 4.4.

## 4. Performance by Driver Tenure

### 4.0 Tenure buckets

In [ ]:
# TODO: define buckets with pd.cut (boundaries after looking at the distribution from section 3),
# show the number of trips per bucket (and how many trips have no bucket)

### 4.1 Does on-time rate differ by driver tenure?

In [ ]:
# TODO: groupby tenure bucket -> on_time_flag mean / delay rate

### 4.2 Does fuel efficiency differ by driver tenure?

In [ ]:
# TODO: groupby tenure bucket -> mean average_mpg (via trips -> driver_id)

In [ ]:
# TODO: chart(s)

### 4.3 Is the difference more than chance?

In [ ]:
# TODO: permutation test on the bucket labels (permutation_spread), for the delay rate and average_mpg

*TODO: interpretation.*

## 5. Do Individual Drivers Stand Out?

In [ ]:
# TODO: per-driver delay rate (minimum trip count, justify the threshold) + permutation test,
# same approach as notebooks 3/4/5 (permutation_spread, seed 42, 5,000 iterations)

In [ ]:
# TODO: plot

*TODO: interpretation.*

## 6. Driver Monthly Metrics

### 6.1 How does driver turnover develop over time?

In [ ]:
# TODO: active drivers per month, hires and terminations 2022-2024 from the drivers table (hire_date / termination_date),
# turnover per year (define it in the text); driver_monthly_metrics only holds the drivers still employed, so it carries no turnover information

*TODO: interpretation - dataset description states ~15% annual turnover; compare.*

### 6.2 What does driver_monthly_metrics contain?

In [ ]:
# TODO: only still-employed drivers (124 x 36 months); compare trips_completed / total_miles with the trips table
# (sums, never average the rates); survivorship caveat for any analysis on this table

*TODO: interpretation.*

## 7. Safety Incidents

### 7.1 What do incident rates look like?

In [ ]:
# TODO: incidents per million miles (overall): count only incidents linked to trips in the cleaned data,
# miles from the same trips; shares of at_fault_flag / preventable_flag / injury_flag; breakdown by incident_type

### 7.2 Do incident rates relate to driver experience?

In [ ]:
# TODO: incidents AND miles driven per tenure bucket -> incident rate per million miles
# (counts alone mislead because buckets differ in exposure); caution: only 170 incidents total (fewer after linking to cleaned trips)

In [ ]:
# TODO: chart(s)

*TODO: interpretation - be explicit about sample size limitations here.*

## 8. Conclusion

*TODO: summarize (findings and ideas for later kept apart); mention notebook 7 as an outlook, no link until it exists.*